# Notebook 06 - OCR, Layout, and Table Extraction

This notebook treats document intelligence as a layout problem: fields, boxes, rows, and page structure, not just plain OCR text.


## What you will learn

- how to represent document layout as explicit geometric objects
- how to extract table-like records from page coordinates
- how to prepare page evidence for later retrieval or answer grounding


In [ ]:
!pip install -q numpy pandas matplotlib Pillow pydantic
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "06_ocr_layout_and_table_extraction"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)

from PIL import Image, ImageDraw

print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Represent a page as boxes

Boxes are the bridge between raw page pixels and later structured outputs.


In [ ]:
page = Image.new("RGB", (900, 1200), "white")
draw = ImageDraw.Draw(page)
boxes = [
    {"label": "invoice_number", "text": "INV-2048", "x1": 80, "y1": 90, "x2": 280, "y2": 130},
    {"label": "date", "text": "2026-03-17", "x1": 600, "y1": 90, "x2": 790, "y2": 130},
    {"label": "item_row", "text": "Sensor kit", "x1": 90, "y1": 300, "x2": 350, "y2": 340},
    {"label": "price_row", "text": "$840.00", "x1": 640, "y1": 300, "x2": 790, "y2": 340},
    {"label": "total", "text": "$1,240.00", "x1": 620, "y1": 980, "x2": 810, "y2": 1020},
]
for box in boxes:
    draw.rectangle([box["x1"], box["y1"], box["x2"], box["y2"]], outline="#d62728", width=3)
    draw.text((box["x1"], box["y1"] - 18), box["label"], fill="black")
plt.figure(figsize=(8, 10))
plt.imshow(page)
plt.axis("off")
plt.show()
layout_df = pd.DataFrame(boxes)
layout_df


## Step 2: Recover table rows

Even a lightweight row grouping heuristic goes a long way when fields are well aligned.


In [ ]:
layout_df["row_bucket"] = (layout_df["y1"] // 60).astype(int)
row_view = layout_df.groupby("row_bucket").agg(
    labels=("label", list),
    texts=("text", list),
).reset_index()
row_view


## Step 3: Turn layout into structured output

Structured extraction should preserve both the value and the evidence that supports it.


In [ ]:
structured_invoice = {
    "invoice_number": "INV-2048",
    "date": "2026-03-17",
    "line_items": [{"description": "Sensor kit", "price": 840.0}],
    "total": 1240.0,
    "evidence": {
        "invoice_number": "page1:80,90,280,130",
        "total": "page1:620,980,810,1020",
    },
}
print(json.dumps(structured_invoice, indent=2))


## Exercises and extensions

1. Add more rows and test whether your grouping heuristic still works.
1. Store the structured invoice object as JSON under NOTEBOOK_ROOT for later retrieval experiments.
